**PREPARING FOR LLAMA FINE TUNING**

Lets say you are building a training and evaluation pipeline for your company's health care chatbot, which is used by hospitals to onboard new patients.

Your task is to create a pipeline to load the MedQuad-MedicalQnADataset to evaluate an LLM on its ability to answer medical questions.

In [2]:
from datasets import load_dataset, Dataset
import yaml
import pandas as pd

In [3]:
# Load the training split of the dataset
ds = load_dataset('keivalya/MedQuad-MedicalQnADataset', split='train')

# Filter for the first 500 samples of the dataset
filtered_ds = Dataset.from_dict(ds[:500])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


As part of a customer service chatbot that your team is building, you are creating a pipeline to preprocess a dataset that will eventually be used to fine-tune a language model so that it can predict the intent of a customer's question and route the requests to the correct team for processing.

You are given a dataset with the customer's question and intent in separate columns, and you want to preprocess the dataset so that you have merged each example containing the question and intent into a single string with your formatted prompt.

In [4]:
def create_intent_example(row):
    # Fill out the columns in the prompt
    row['intent_example'] = "Query: {instruction}\nIntent: {intent}"
    return row

# Call the ds method to apply our preprocessing function to all rows
processed_dataset = filtered_ds.map(create_intent_example)
# Print the intent_example in the first row of the processed data
print(processed_dataset[0]['intent_example'])

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Query: {instruction}
Intent: {intent}


As part of your customer service chatbot project, you now have prepared a dataset for fine-tuning a Llama model. The next step is to save the dataset so that you can reload it later without having to repeat the preprocessing steps. This will allow your team to reuse the dataset across multiple experiments and iterations.

In [5]:
from datasets import load_from_disk

# Save the dataset to disk
ds.save_to_disk('preprocessed_dataset')

# Load the dataset from disk
ds_preprocessed = load_from_disk('preprocessed_dataset')

# Print the first element of the loaded dataset
print(ds_preprocessed[0])

Saving the dataset (0/1 shards):   0%|          | 0/16407 [00:00<?, ? examples/s]

{'qtype': 'susceptibility', 'Question': 'Who is at risk for Lymphocytic Choriomeningitis (LCM)? ?', 'Answer': 'LCMV infections can occur after exposure to fresh urine, droppings, saliva, or nesting materials from infected rodents.  Transmission may also occur when these materials are directly introduced into broken skin, the nose, the eyes, or the mouth, or presumably, via the bite of an infected rodent. Person-to-person transmission has not been reported, with the exception of vertical transmission from infected mother to fetus, and rarely, through organ transplantation.'}


 You've successfully saved and reloaded the preprocessed dataset: this will ensure that it can be reused in your customer service chatbot project for new experiments and training tasks.

You're fine-tuning a pre-trained Llama model for a customer who requires specific configurations.

In [6]:
config_dict = {
    # Define the model
    "model":{"_component_": "torchtune.models.llama3_2.llama3_2_1b"},
    # Define the batch size
    "batch_size":8,
    # Define the device type
    "device":"cuda",
    "epochs": 15,
    "optimizer": {"_component_": "bitsandbytes.optim.PagedAdamW8bit", "lr": 3e-05},
    "dataset": {"_component_": "custom_dataset"},
    "output_dir": "/tmp/finetune_results"
}

The customer has now asked you for a modification in the requirements. This time, they'd like to increase the number of parameters and use the Llama 3.2 model with 3B parameters. You make this modification to your dictionary, and then save it as a YAML file.

In [7]:
config_dict = {
    # Update the model
    "model":{"_component_":"torchtune.models.llama3_2.llama3_2_3b"},
    "batch_size": 8,
    "device": "cuda",
    "optimizer": {"_component_": "bitsandbytes.optim.PagedAdamW8bit", "lr": 3e-05},
    "dataset": {"_component_": "custom_dataset"},
    "output_dir": "/tmp/finetune_results"
}

# Save the updated configuration to a new YAML file
with open("custom_recipe.yaml", "w") as yaml_file:
    yaml.dump(config_dict, yaml_file)

**Fine-tuning with SFTTrainer on Hugging Face**

You are tasked with working with the Llama model used in a customer service chatbot by fine-tuning it on customer service data purpose built for question-answering. To ensure the best performance out of these models, your team will fine-tune a Llama model for this task using the bitext dataset.

In [8]:
from transformers import TrainingArguments

training_arguments = TrainingArguments(
    learning_rate=2e-3,
    warmup_steps=0.03,
    num_train_epochs=3,
    output_dir='/tmp',
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    save_steps=10,
    logging_steps=2,
    lr_scheduler_type='constant',
    report_to='none'
)

print("TrainingArguments created successfully!")

TrainingArguments created successfully!


In [9]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load the Llama model and tokenizer
model_name = 'Maykeye/TinyLLama-v0'
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Set the padding token for the tokenizer
tokenizer.pad_token = tokenizer.eos_token

Loading weights:   0%|          | 0/75 [00:00<?, ?it/s]

In [10]:
!pip install trl

In [11]:
bitext = pd.read_csv('/content/Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv')
bitext_20 = bitext.head(20)
# Import the supervised fine-tuning class
from trl import SFTTrainer
from datasets import Dataset # Import Dataset class
# Define a function to format the dataset rows
def format_row(row):
  # Combine the 'Question' and 'Answer' into a single string
  # This format is common for fine-tuning Q&A models
  row['text'] = f"Question: {row['instruction']}\nAnswer: {row['response']}"
  return row

# Apply the formatting function to the filtered_ds to create a new column
formatted_train_dataset_df = bitext_20.apply(format_row, axis=1)

# Convert the Pandas DataFrame to a datasets.Dataset object
formatted_train_dataset = Dataset.from_pandas(formatted_train_dataset_df)

In [12]:
from trl import SFTTrainer

# Instantiate fine-tuning class
trainer = SFTTrainer(
  	# Pass necessary arguments
    model=model,
    train_dataset=formatted_train_dataset,
    args=training_arguments,

)

# Start training
trainer.train()

Adding EOS to train dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': None}.


Step,Training Loss
2,8.275652
4,6.859794
6,6.191967
8,5.608755
10,5.878631
12,5.410746
14,5.227186
16,5.152037
18,4.536401
20,4.094830


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=60, training_loss=3.6272515575091044, metrics={'train_runtime': 20.7759, 'train_samples_per_second': 2.888, 'train_steps_per_second': 2.888, 'total_flos': 248603561856.0, 'train_loss': 3.6272515575091044})

Lets ask some questions to this model and evaluate this using ROGUE score

In [13]:
from huggingface_hub import login
login()

In [14]:
from datasets import load_dataset

dataset = load_dataset("Softage-AI/sft-conversational_dataset")

# Take 10 samples (usually from test split)
test_samples = dataset['train'].select(range(10))  # or 'test' if available

In [15]:
questions = [row['Query'] for row in test_samples]
reference_answers = [row['Answer'] for row in test_samples]

In [16]:
test_answers = []

for q in questions:
    prompt = f"Question: {q}\nAnswer:"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,        # or False for deterministic
        temperature=0.7
    )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Remove prompt from output
    answer = answer.replace(prompt, "").strip()

    test_answers.append(answer)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [17]:
for i in range(10):
    print(f"\nQ{i+1}: {questions[i]}")
    print("Generated:", test_answers[i])
    print("Actual   :", reference_answers[i])


Q1: When is the best time to visit Alaska?
Generated: I'm here to canceling your order with the order number {{Order Number}}. I'm here to help you with what '{{}}' or '{{}}' or '{{}}'. Please select it to assist you.

1. Look for the purchase with the order number {{Order Number}} and click on it to select it to select it.
3. Locate the ' Portal{{Custom need to assist you.

1. Confirm the details: Look
Actual   : The best time to visit Alaska is during the summer between May and September. The temperature is warm and pleasant, there are 16–24 hours of daylight and the scenery is breathtaking.

Here is a more detailed breakdown of the best time to visit Alaska for different activities:
1. The Northern Lights: The northern lights are visible from August 20 through April 20. The best time for a winter aurora vacation is February and March [1].
2. Wildlife viewing: June to August is the best time to see whales, bears, and other wildlife [1].
3. Hiking and camping: May to mid-October is t

In [18]:
# Install the necessary dependency for ROUGE evaluation
!pip install rouge_score
!pip install evaluate

# Import the evaluation library from Hugging Face
import evaluate

# Instantiate your evaluate library and load the ROUGE metric
rouge_evaluator = evaluate.load('rouge')

# Fill in the method, and place your reference answers and test answers
results = rouge_evaluator.compute(predictions=test_answers, references= reference_answers)

# Extract the ROUGE1 score from the results dictionary
final_score = results['rouge1']
print(final_score)

0.13787279526515012


 ROUGE is one of many evaluation metrics available for LLMs, but it is often used since it is easy to understand if you have reference text to compare against.

Lets fine-tuning the Maykeye/TinyLLama-v0 language model to answer customer service questions using the bitext dataset.

In [19]:
# Import LoRA configuration class
from peft import LoraConfig

# Instantiate LoRA configuration with values
lora_config = LoraConfig(
  	r=12,
    lora_alpha=8,
  	task_type="CAUSAL_LM",
    lora_dropout=0.05,
    bias="none",
    target_modules=['q_proj', 'v_proj']
)

trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_train_dataset, # Use the newly formatted dataset
    args=training_arguments,
  	# Pass the lora_config to trainer
  	peft_config= lora_config,
)

Adding EOS to train dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

The Llama models are quite good at question answering, and should work well for this customer service task. Lets say, you don't have the compute capacity to conduct regular fine-tuning, and must use LoRA fine-tuning techniques using the bitext dataset.

In [ ]:
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer
from transformers import AutoModelForCausalLM, AutoTokenizer # Re-import model and tokenizer to ensure a fresh base model

# Load the Llama model and tokenizer again to get a fresh, un-adapted model
model_name = 'Maykeye/TinyLLama-v0'
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

peft_config = LoraConfig(
    # Set rank parameter
  	r=2,
  	# Set scaling factor
    lora_alpha=4,
  	# Set the type of task
  	task_type="CAUSAL_LM",
    lora_dropout=0.05,
    bias="none",
    target_modules=['q_proj', 'v_proj']
)

trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_train_dataset, # Use the newly formatted dataset
    args=training_arguments,
  	# Pass the newly defined peft_config to trainer
  	peft_config=peft_config
)

trainer.train()

Lets also say we have been figuring out how to reduce the model's GPU memory usage without significantly affecting performance. This will allow the us to switch to a cheaper compute cluster and save the company a lot of money.

You decide to test if you can load your model with 8-bit quantization and maintain a reasonable performance.

In [20]:
# Uninstall any existing versions of bitsandbytes to prevent conflicts
!pip uninstall -y bitsandbytes transformers accelerate

# Install the latest compatible version of bitsandbytes for Colab's CUDA 12.x
# Also update transformers and accelerate to ensure compatibility
!pip install bitsandbytes transformers accelerate


Found existing installation: bitsandbytes 0.39.1
Uninstalling bitsandbytes-0.39.1:
  Successfully uninstalled bitsandbytes-0.39.1
Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
  Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl (60.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 36.8 MB/s eta 0:00:00


In [20]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import BitsAndBytesConfig

model_name = "Maykeye/TinyLLama-v0"
bnb_config = BitsAndBytesConfig(load_in_8bit=True)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    low_cpu_mem_usage=True
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

Loading weights:   0%|          | 0/75 [00:00<?, ?it/s]

Now lets say one of the biggest customer complaints you receive is that the bot answers questions very slowly and sometimes produces weird answers.

You suspect this might have to do with quantizing to 4-bit without normalizing. In your investigation, you also suspect that the speed trade-off comes from the inference computations, which are using 32-bit floats.

You want to adjust the quantization configurations to improve the inference speed of your model.

In [22]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_name="Maykeye/TinyLLama-v0"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
  	# Set quantization type to normalized 4-bit
    bnb_4bit_quant_type="nf4",
  	# Set compute data type to be bfloat16
    bnb_4bit_compute_dtype=torch.bfloat16

)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    low_cpu_mem_usage=True
)

Loading weights:   0%|          | 0/75 [00:00<?, ?it/s]

We've improved inference times and made your bot's responses more stable, driving higher app retention and a better customer experience.

In [28]:
import nbformat
nb = nbformat.read("notebook.ipynb", as_version=nbformat.NO_CONVERT)
if "widgets" in nb.metadata and "state" not in nb.metadata.widgets:
  nb.metadata["widgets"] = {"state": {}}
nbformat.write(nb, "notebook_fixed.ipynb")

FileNotFoundError: [Errno 2] No such file or directory: 'notebook.ipynb'